# Step 5 — Monthly Time-Series Aggregation (Day/Night, Peak/Off-peak, Weekend/Holiday)

**Input**: `output/processed_hourly_bike_counts_time_categories.csv.gz` (output of Step 4)
**Output**:
- `output/monthly_day_night_summary.csv`
- `output/monthly_time_period_summary.csv`
- `output/monthly_temporal_ratios.csv`

## Methodology notes

- **A. day/night**: weighted by `daylight_fraction` (`day_hours_weighted` = the sum of
  `daylight_fraction` across every record in a month, i.e. "how many equivalent daylight hours
  existed this month"), rather than comparing `total_day_count` and `total_night_count` directly.
  Every month naturally has a different number of daylight/night hours (long summer days, short
  winter days), so comparing raw totals would just be confounded by season length. What matters is
  traffic density -- average count per daylight hour vs average count per night hour.
- **B. time_period**: descriptive statistics computed per month for each of the three groups --
  `weekday_peak` / `weekday_offpeak` / `weekend_public_holiday`.
- **C. ratios**:
  - `weekday_weekend_ratio` uses "weekday" as the **weighted combination** of `weekday_peak` +
    `weekday_offpeak` (weighted by total_count / number_of_records, not a plain average of the two
    group averages, to avoid bias from peak and off-peak having different row counts).
  - `night_share` / `peak_share` / `weekend_share` are all shares of a **summed quantity** (total
    count or weighted count), not shares of row counts.


In [1]:
import pandas as pd
import numpy as np

INPUT_PATH = "output/processed_hourly_bike_counts_time_categories.csv.gz"
OUT_DAY_NIGHT = "output/monthly_day_night_summary.csv"
OUT_TIME_PERIOD = "output/monthly_time_period_summary.csv"
OUT_RATIOS = "output/monthly_temporal_ratios.csv"

In [2]:
df = pd.read_csv(
    INPUT_PATH,
    dtype={'station_id': 'int64', 'month': 'str', 'time_period': 'str',
           'bike_count_hourly': 'float64', 'daylight_fraction': 'float64',
           'day_count_weighted': 'float64', 'night_count_weighted': 'float64'},
    usecols=['station_id', 'month', 'time_period', 'bike_count_hourly',
             'daylight_fraction', 'day_count_weighted', 'night_count_weighted'],
)
print(f"Loaded: {len(df):,} rows")

Loaded: 3,227,633 rows


## A. Monthly day/night summary

In [3]:
day_night = df.groupby('month', observed=True).agg(
    total_day_count=('day_count_weighted', 'sum'),
    total_night_count=('night_count_weighted', 'sum'),
    day_hours_weighted=('daylight_fraction', 'sum'),
).reset_index()

# night_hours_weighted = sum(1 - daylight_fraction)
night_hours = df.groupby('month', observed=True)['daylight_fraction'].apply(lambda x: (1 - x).sum())
day_night['night_hours_weighted'] = day_night['month'].map(night_hours)

day_night['avg_day_count_per_hour'] = day_night['total_day_count'] / day_night['day_hours_weighted']
day_night['avg_night_count_per_hour'] = day_night['total_night_count'] / day_night['night_hours_weighted']
day_night['day_night_ratio'] = day_night['avg_day_count_per_hour'] / day_night['avg_night_count_per_hour']

day_night = day_night.sort_values('month').reset_index(drop=True)
day_night.to_csv(OUT_DAY_NIGHT, index=False)
print(f"Written: {OUT_DAY_NIGHT}")
day_night

Written: output/monthly_day_night_summary.csv


,month,total_day_count,total_night_count,day_hours_weighted,night_hours_weighted,avg_day_count_per_hour,avg_night_count_per_hour,day_night_ratio
0,2025-01,1.709606e+06,1.571485e+06,80175.723193,156652.276807,21.323238,10.031676,2.125591
1,2025-02,2.258030e+06,1.156428e+06,88405.113459,127291.886541,25.541846,9.084854,2.811476
2,2025-03,3.844615e+06,8.540680e+05,117578.616594,119906.383406,32.698250,7.122790,4.590652
3,2025-04,5.240349e+06,9.072551e+05,134708.731065,95474.268935,38.901330,9.502614,4.093750
4,2025-05,5.913616e+06,4.412442e+05,158285.099230,80160.900770,37.360534,5.504482,6.787294
5,2025-06,6.334309e+06,2.906548e+05,162192.418376,67582.581624,39.054286,4.300736,9.080838
6,2025-07,6.294422e+06,4.154932e+05,161665.782366,74644.217634,38.934781,5.566315,6.994714
7,2025-08,5.197704e+06,6.789165e+05,144696.701933,91196.298067,35.921375,7.444562,4.825183
8,2025-09,4.692469e+06,1.255881e+06,119910.884242,107603.115758,39.132971,11.671417,3.352889
9,2025-10,3.115843e+06,1.417192e+06,102765.708007,131604.291993,30.319874,10.768582,2.815586


## B. Monthly time-period summary

In [4]:
tp_summary = df.groupby(['month', 'time_period'], observed=True).agg(
    total_count=('bike_count_hourly', 'sum'),
    number_of_records=('bike_count_hourly', 'size'),
    average_hourly_count=('bike_count_hourly', 'mean'),
    median_hourly_count=('bike_count_hourly', 'median'),
    standard_deviation=('bike_count_hourly', 'std'),
).reset_index()
tp_summary['coefficient_of_variation'] = tp_summary['standard_deviation'] / tp_summary['average_hourly_count']

tp_summary = tp_summary.sort_values(['month', 'time_period']).reset_index(drop=True)
tp_summary.to_csv(OUT_TIME_PERIOD, index=False)
print(f"Written: {OUT_TIME_PERIOD}  (rows: {len(tp_summary)}, expected 14 months x 3 time_periods = 42)")
tp_summary.head(6)

Written: output/monthly_time_period_summary.csv  (rows: 42, expected 14 months x 3 time_periods = 42)


,month,time_period,total_count,number_of_records,average_hourly_count,median_hourly_count,standard_deviation,coefficient_of_variation
0,2025-01,weekday_offpeak,1484752.0,126140,11.770668,3.0,22.190469,1.885235
1,2025-01,weekday_peak,1333508.0,42078,31.691335,13.0,51.304233,1.618873
2,2025-01,weekend_public_holiday,462831.0,68610,6.745824,2.0,13.720705,2.033955
3,2025-02,weekday_offpeak,1465900.0,115547,12.686612,3.0,23.273429,1.834487
4,2025-02,weekday_peak,1329005.0,38518,34.503479,14.0,54.189689,1.570557
5,2025-02,weekend_public_holiday,619553.0,61632,10.052457,3.0,18.974486,1.887547


## C. Monthly ratios

In [5]:
pivot_avg = tp_summary.pivot(index='month', columns='time_period', values='average_hourly_count')
pivot_total = tp_summary.pivot(index='month', columns='time_period', values='total_count')
pivot_n = tp_summary.pivot(index='month', columns='time_period', values='number_of_records')

ratios = pd.DataFrame(index=pivot_avg.index)

# peak_offpeak_ratio
ratios['peak_offpeak_ratio'] = pivot_avg['weekday_peak'] / pivot_avg['weekday_offpeak']

# weekday_weekend_ratio: weekday = weekday_peak + weekday_offpeak, weighted combination
weekday_total = pivot_total['weekday_peak'] + pivot_total['weekday_offpeak']
weekday_n = pivot_n['weekday_peak'] + pivot_n['weekday_offpeak']
weekday_avg = weekday_total / weekday_n
ratios['weekday_weekend_ratio'] = weekday_avg / pivot_avg['weekend_public_holiday']

# night_share / peak_share / weekend_share
monthly_total_count = pivot_total.sum(axis=1)
day_night_idx = day_night.set_index('month')
ratios['night_share'] = (day_night_idx['total_night_count'] /
                          (day_night_idx['total_day_count'] + day_night_idx['total_night_count']))
ratios['peak_share'] = pivot_total['weekday_peak'] / monthly_total_count
ratios['weekend_share'] = pivot_total['weekend_public_holiday'] / monthly_total_count

ratios = ratios.reset_index().sort_values('month').reset_index(drop=True)
ratios.to_csv(OUT_RATIOS, index=False)
print(f"Written: {OUT_RATIOS}")
ratios

Written: output/monthly_temporal_ratios.csv


,month,peak_offpeak_ratio,weekday_weekend_ratio,night_share,peak_share,weekend_share
0,2025-01,2.692399,2.483554,0.478952,0.406422,0.141060
1,2025-02,2.719676,1.804641,0.338686,0.389229,0.181450
2,2025-03,2.561766,1.740220,0.181768,0.361455,0.214860
3,2025-04,1.827042,1.998402,0.147579,0.302603,0.200354
4,2025-05,1.852132,1.637174,0.069434,0.285718,0.251522
5,2025-06,1.856007,1.709727,0.043873,0.295711,0.226315
6,2025-07,1.861585,1.542180,0.061922,0.312443,0.184130
7,2025-08,1.828151,1.548946,0.115528,0.289538,0.235341
8,2025-09,1.786410,1.496857,0.211131,0.300249,0.195457
9,2025-10,1.873798,2.080975,0.312636,0.312707,0.186564
